# PatchSight: full extraction run on Kaggle

Serves Qwen3.8-27B (4-bit GGUF, Q4_K_M) with llama.cpp on two T4 GPUs and runs PatchSight's
extractor on the Dunlap Go benchmark. Groq is used for the dev split; this notebook is for
full runs and, later, the PyPI backfill.

**Before running**

1. *Settings → Accelerator:* **GPU T4 x2**. *Settings → Internet:* **on**.
2. *Add-ons → Secrets:* add **`GITHUB_TOKEN`** and attach it to this notebook. A GitHub token
   with no scopes is enough: it only reads public commits.
3. Use **Save Version → Save & Run All (Commit)**, so the run continues in the background
   for up to 12 hours.

**Output:** `/kaggle/working/patchsight-kaggle-run.zip`, holding the model responses, the
reports and `run-info.json`. Import it on your machine with
`python -m patchsight.pipeline import-kaggle-run patchsight-kaggle-run.zip`, then regenerate
the reports with `dunlap-extract --role kaggle_extractor`, which replays from the cache.

**Expected time:** build about 10 minutes, model download about 10 minutes, extraction a few
hours for all 280 reports.

## 1. Pinned versions

In [ ]:
PATCHSIGHT_REPO = "https://github.com/professor3333/PatchSight.git"
PATCHSIGHT_REF = "main"
LLAMA_CPP_TAG = "b12024"  # released 2026-10-31
LLAMA_SOURCE = "prebuilt"  # "prebuilt" (official CUDA 12.8 binaries) or "build" (compile here)
# Published digests of the official release assets.
LLAMA_SHA256 = "2214b85fa76676e395c4928bc5cbcb3536e2a33f83783fffdaca3ce0c11c8acc"
CUDART_SHA256 = "2bf29bb45a323a3633ddb323a906457f03041d93ed30ad8cee67150f2dac54cb"
LLAMA_ARCHIVES = {
    f"llama-{LLAMA_CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz": LLAMA_SHA256,
    f"cudart-llama-{LLAMA_CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz": CUDART_SHA256,
}
MODEL_REPO = "ggml-org/Qwen3.8-27B-GGUF"
MODEL_REVISION = "c1869413dece09109e55b7dcc2c77ef382b0d0bd"
MODEL_FILE = "Qwen3.8-27B-Q4_K_M.gguf"
MODEL_SHA256 = "d15d4b6de55f662ac40e867702d31afe738d032fa06b829b2b1a21a49d4a48ee"
SPLIT = "all"  # dev, test or all
WORK = "/tmp/patchsight"

## 2. Check the GPUs and disk

In [ ]:
import os
import shutil
import subprocess

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
assert gpus >= 2, "set Accelerator to GPU T4 x2"
os.makedirs(WORK, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"free disk at {WORK}: {free_gb:.0f} GB")
assert free_gb > 30, "need about 30 GB free for the model and the build"

## 3. Get llama.cpp

The official prebuilt CUDA binaries are used by default, verified against their published
SHA-256. If they cannot see the GPUs, llama.cpp is built from source for the T4 (compute
capability 7.5). Either way, any failure prints the log that explains it.

In [ ]:
import glob
import hashlib
import tarfile
import urllib.request


def sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()


smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(
    next(
        (line for line in smi.splitlines() if "CUDA Version" in line), "nvidia-smi header not found"
    )
)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)

LLAMA_SERVER = None
server_env = dict(os.environ)
if LLAMA_SOURCE == "prebuilt":
    base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_CPP_TAG}/"
    unpacked = f"{WORK}/llama-prebuilt"
    for name, expected in LLAMA_ARCHIVES.items():
        archive_path = f"{WORK}/{name}"
        if not os.path.exists(archive_path):
            urllib.request.urlretrieve(base + name, archive_path)
        assert sha256(archive_path) == expected, f"{name} does not match its published SHA-256"
        with tarfile.open(archive_path) as archive:
            archive.extractall(unpacked, filter="data")
    library_dirs = sorted(
        {os.path.dirname(p) for p in glob.glob(f"{unpacked}/**/*.so*", recursive=True)}
    )
    server_env["LD_LIBRARY_PATH"] = ":".join([*library_dirs, os.environ.get("LD_LIBRARY_PATH", "")])
    candidates = glob.glob(f"{unpacked}/**/llama-server", recursive=True)
    if candidates:
        check = subprocess.run(
            [candidates[0], "--list-devices"], env=server_env, capture_output=True, text=True
        )
        print((check.stdout + check.stderr)[-1500:])
        if check.returncode == 0 and "CUDA" in check.stdout + check.stderr:
            LLAMA_SERVER = candidates[0]
    print("prebuilt binaries usable:", LLAMA_SERVER is not None)

In [ ]:
def run_logged(command: list[str], log_path: str, cwd: str) -> None:
    with open(log_path, "w") as log:
        result = subprocess.run(command, cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        with open(log_path) as log:
            print("".join(log.readlines()[-80:]))
        raise RuntimeError(f"{command[0]} failed; the last 80 lines of {log_path} are above")


if LLAMA_SERVER is None:
    for tool in (["nvcc", "--version"], ["cmake", "--version"]):
        try:
            found = subprocess.run(tool, capture_output=True, text=True).stdout.strip()
            print(found.splitlines()[-1] if found else f"{tool[0]}: no output")
        except FileNotFoundError:
            print(f"{tool[0]}: not installed")
    source = f"{WORK}/llama.cpp"
    if not os.path.isdir(source):
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "--depth",
                "1",
                "--branch",
                LLAMA_CPP_TAG,
                "https://github.com/ggml-org/llama.cpp.git",
                source,
            ],
            check=True,
        )
    run_logged(
        [
            "cmake",
            "-B",
            "build",
            "-DGGML_CUDA=ON",
            "-DCMAKE_CUDA_ARCHITECTURES=75",
            "-DGGML_NATIVE=OFF",
            "-DLLAMA_CURL=OFF",
            "-DCMAKE_BUILD_TYPE=Release",
        ],
        f"{WORK}/cmake-configure.log",
        source,
    )
    run_logged(
        [
            "cmake",
            "--build",
            "build",
            "--config",
            "Release",
            "-j",
            str(os.cpu_count()),
            "--target",
            "llama-server",
        ],
        f"{WORK}/cmake-build.log",
        source,
    )
    LLAMA_SERVER = f"{source}/build/bin/llama-server"
    server_env = dict(os.environ)
print("llama-server:", LLAMA_SERVER)
print(
    subprocess.run(
        [LLAMA_SERVER, "--version"], env=server_env, capture_output=True, text=True
    ).stderr.strip()[-300:]
)

## 4. Download the model and verify it against the pinned SHA-256

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    MODEL_REPO, MODEL_FILE, revision=MODEL_REVISION, local_dir=f"{WORK}/models"
)
assert sha256(model_path) == MODEL_SHA256, "the model file does not match the pinned SHA-256"
print("model verified:", model_path)

## 5. Start llama-server

Layers are split across both GPUs. The context of 12,288 tokens covers the largest prompt
(about 5,500 tokens), the answer (1,500) and one repair round. Thinking mode is switched off
per request through the chat template (see `configs/llm.toml`).

In [ ]:
import secrets
import time
import urllib.request

os.environ["LLAMA_SERVER_KEY"] = secrets.token_hex(16)
server_log = open(f"{WORK}/llama-server.log", "w")
server = subprocess.Popen(
    [
        LLAMA_SERVER,
        "-m",
        model_path,
        "--host",
        "127.0.0.1",
        "--port",
        "8080",
        "--n-gpu-layers",
        "999",
        "--split-mode",
        "layer",
        "--ctx-size",
        "12288",
        "--parallel",
        "1",
        "--jinja",
        "--seed",
        "20261114",
        "--api-key",
        os.environ["LLAMA_SERVER_KEY"],
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=server_env,
)
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError("llama-server exited; read llama-server.log")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5) as response:
            if response.status == 200:
                break
    except OSError:
        pass
    time.sleep(5)
else:
    raise RuntimeError("llama-server did not become healthy within 15 minutes")
print("llama-server is up")

## 6. Smoke test: one schema-constrained answer with thinking off

In [ ]:
import json

schema = {
    "type": "object",
    "additionalProperties": False,
    "required": ["answer"],
    "properties": {"answer": {"type": "string"}},
}
body = {
    "model": "local",
    "temperature": 0,
    "seed": 1,
    "max_tokens": 50,
    "messages": [{"role": "user", "content": "Reply with the word ready."}],
    "response_format": {
        "type": "json_schema",
        "json_schema": {"name": "smoke", "schema": schema, "strict": True},
    },
    "chat_template_kwargs": {"enable_thinking": False},
}
request = urllib.request.Request(
    "http://127.0.0.1:8080/v1/chat/completions",
    data=json.dumps(body).encode(),
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['LLAMA_SERVER_KEY']}",
    },
)
try:
    with urllib.request.urlopen(request, timeout=300) as response:
        reply = json.load(response)
except OSError as error:
    with open(f"{WORK}/llama-server.log") as handle:
        print("".join(handle.readlines()[-40:]))
    raise RuntimeError(
        "the smoke test failed; if the log above mentions 'no kernel image', set "
        "LLAMA_SOURCE = 'build' in section 1 and run all cells again"
    ) from error
content = reply["choices"][0]["message"]["content"]
print(content, reply["usage"])
assert "answer" in json.loads(content)

## 7. Install PatchSight and load the GitHub token

In [ ]:
%%bash -s "$PATCHSIGHT_REPO" "$PATCHSIGHT_REF" "$WORK"
set -euo pipefail
cd "$3"
[ -d repo ] || git clone -q "$1" repo
cd repo
git fetch -q origin
git checkout -q "$2"
git log -1 --format='PatchSight %H %s'
pip install -q uv
uv sync -q --all-extras --frozen

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
print("GITHUB_TOKEN loaded")  # never print the value

## 8. Run extraction

`fetch-dunlap` checks out the pinned dataset commit and verifies its hashes. The extraction
fetches each fix commit from GitHub (about 1,800 requests) and calls the local model once per
report, plus one repair attempt where needed. The dev and test reports are then built from the
cache, with no further model calls.

In [ ]:
REPO = f"{WORK}/repo"


def pipeline(*args: str) -> None:
    command = ["uv", "run", "--frozen", "python", "-m", "patchsight.pipeline", *args]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)


tag = "extract-v1-qwen3.8-27b-q4km"
pipeline("fetch-dunlap")
pipeline(
    "dunlap-extract",
    "--role",
    "kaggle_extractor",
    "--split",
    SPLIT,
    "--out",
    f"docs/reports/extraction-dunlap-{SPLIT}-{tag}",
)
for part in ("dev", "test"):
    if part != SPLIT:
        pipeline(
            "dunlap-extract",
            "--role",
            "kaggle_extractor",
            "--split",
            part,
            "--out",
            f"docs/reports/extraction-dunlap-{part}-{tag}",
        )

## 9. Package the results

In [ ]:
import glob
import zipfile

run_info = {
    "patchsight_commit": subprocess.run(
        ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True
    ).stdout.strip(),
    "llama_cpp_tag": LLAMA_CPP_TAG,
    "model_repo": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "model_file": MODEL_FILE,
    "model_sha256": MODEL_SHA256,
    "gpus": subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    .stdout.strip()
    .splitlines(),
    "split": SPLIT,
}
archive = "/kaggle/working/patchsight-kaggle-run.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.writestr("run-info.json", json.dumps(run_info, indent=2))
    with open(f"{WORK}/llama-server.log") as handle:
        bundle.writestr("llama-server.log", "".join(handle.readlines()[-300:]))
    for path in glob.glob(f"{REPO}/.cache/llm/**/*", recursive=True):
        if os.path.isfile(path):
            bundle.write(path, os.path.relpath(path, REPO))
    for path in glob.glob(f"{REPO}/docs/reports/extraction-dunlap-*-{tag}.*"):
        bundle.write(path, os.path.relpath(path, REPO))
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")
print(json.dumps(run_info, indent=2))

In [ ]:
server.terminate()